# 1. Feedforward Neural Networks

This notebook builds neural networks from first principles, then with PyTorch:
- **From scratch**: forward pass, backpropagation, gradient descent with NumPy
- **With PyTorch**: `nn.Module`, automatic differentiation, training on MNIST
- Activation functions, loss functions, and optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)

## 1.1 A Neural Network from Scratch (NumPy)

We build a 2-layer network for binary classification on the XOR problem.

Architecture: Input (2) -> Hidden (4, ReLU) -> Output (1, Sigmoid)

Forward pass:
$$z_1 = W_1 x + b_1, \quad h = \text{ReLU}(z_1), \quad z_2 = W_2 h + b_2, \quad \hat{y} = \sigma(z_2)$$

In [ ]:
# XOR data
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

# Activation functions
sigmoid = lambda z: 1 / (1 + np.exp(-z))
relu = lambda z: np.maximum(0, z)
relu_deriv = lambda z: (z > 0).astype(float)

# Initialize weights
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))

lr = 0.5
losses = []

for epoch in range(5000):
    # Forward
    z1 = X @ W1 + b1
    h = relu(z1)
    z2 = h @ W2 + b2
    y_hat = sigmoid(z2)
    
    # Binary cross-entropy loss
    loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))
    losses.append(loss)
    
    # Backward
    dz2 = y_hat - y                       # (4, 1)
    dW2 = h.T @ dz2 / 4
    db2 = np.mean(dz2, axis=0, keepdims=True)
    dh = dz2 @ W2.T                       # (4, 4)
    dz1 = dh * relu_deriv(z1)
    dW1 = X.T @ dz1 / 4
    db1 = np.mean(dz1, axis=0, keepdims=True)
    
    # Update
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

print("Predictions after training:")
print(np.round(y_hat, 3).flatten(), "(target:", y.flatten(), ")")

plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss (XOR from scratch)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.2 PyTorch Basics

PyTorch provides:
- **Tensors** with GPU support
- **Autograd** for automatic differentiation
- `nn.Module` for defining models

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1.3 MNIST Classification with PyTorch

Architecture: 784 -> 256 (ReLU) -> 128 (ReLU) -> 10 (Softmax)

In [ ]:
# Data loading
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

class FeedForwardNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        return self.net(x)

model = FeedForwardNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Training loop
n_epochs = 5
train_losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.4f}")

# Evaluate
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()

print(f"\nTest accuracy: {correct / len(test_data):.4f}")

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
images_vis, labels_vis = next(iter(test_loader))
model.eval()
with torch.no_grad():
    preds_vis = model(images_vis.to(device)).argmax(dim=1).cpu()

for i, ax in enumerate(axes.flat):
    ax.imshow(images_vis[i].squeeze(), cmap='gray')
    color = 'green' if preds_vis[i] == labels_vis[i] else 'red'
    ax.set_title(f'Pred: {preds_vis[i].item()}', color=color)
    ax.axis('off')
plt.suptitle('MNIST Predictions (green=correct, red=wrong)')
plt.tight_layout()
plt.show()

## Key Takeaways

- A neural network is a composition of affine transformations and nonlinear activations
- **Backpropagation** is just the chain rule applied systematically
- PyTorch's **autograd** handles gradient computation automatically
- A simple feedforward network achieves ~98% on MNIST
- Key hyperparameters: learning rate, hidden size, number of layers, batch size